[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/main/notebooks/case_studies/research_assistant/crewai.ipynb)

# Research assistant — CrewAI

**Task.** Answer a user-supplied research question using specialist CrewAI agents and deterministic safety controls.

**Read this notebook as:** configure a provider → adapt the shared model → define planner, extractor, writer and critic agents/tasks → run the paper's eight case-study stages.


In [ ]:
import os

# 1. Choose the model provider; then use Run All. No terminal command is needed.
MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "research-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = os.getenv(
    "AGENTIC_TUTORIAL_LOCAL_MODEL_PATH",
    "models/local/Qwen3-0.6B-Q8_0.gguf",
)
API_MODEL_NAME = "gemini-3.1-flash-lite"
SAVE_API_CREDENTIAL = True  # saves hidden input to ignored .private/ storage
RESEARCH_QUESTION = "Which interventions reduce household food waste, and under what conditions?"
# Mock runtime is under one minute on CPU; local and API runs can be slower.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "crewai==1.15.2", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import asyncio
import contextlib
import io
import json
import os
import re
import tempfile
import threading
from pathlib import Path
from typing import ClassVar, Literal

import appdirs

os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")
appdirs.user_data_dir = lambda *args, **kwargs: str(
    Path(tempfile.gettempdir()) / "agentic-ai-tutorial-crewai"
)

from crewai import Agent, Crew, Process, Task  # noqa: E402
from crewai.llms.base_llm import BaseLLM  # noqa: E402
from pydantic import BaseModel, ConfigDict, Field  # noqa: E402

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))

from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.notebook import prepare_gemini_api_key  # noqa: E402
from agentic_tutorial.safety import (  # noqa: E402
    PolicyOutcome,
    SafetyEngine,
    SafetyPolicy,
    UntrustedContent,
)
from agentic_tutorial.schemas import CriticDecision, Message, MessageRole  # noqa: E402

catalogue = json.loads((ROOT / "data/research_assistant/evidence_catalogue.json").read_text())
fixture_path = ROOT / "data/research_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())
if MODEL_PROVIDER == "api":
    prepare_gemini_api_key(ROOT, save=SAVE_API_CREDENTIAL)

## Native CrewAI agents and tasks

CrewAI `Agent`, `Task` and `Crew` objects perform the four model-mediated stages. Ordinary Python remains responsible for bounded retrieval, untrusted-content screening, deterministic claim validation, termination and reporting. This keeps the framework demonstration native without delegating safety-critical control to the model.


In [ ]:
# --- Declarations: typed outputs, shared provider adapter, agents and workflow stages ---
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class QueryPlan(Strict):
    schema_id: ClassVar[str] = "research.query.v1"
    queries: tuple[str, ...] = Field(min_length=1, max_length=4)


class Claim(Strict):
    source_id: str
    claim: str
    stance: Literal["supporting", "conflicting"]


class Ledger(Strict):
    schema_id: ClassVar[str] = "research.ledger.v1"
    claims: tuple[Claim, ...]


class Synthesis(Strict):
    schema_id: ClassVar[str] = "research.synthesis.v1"
    answer: str
    source_ids: tuple[str, ...]
    outcome: Literal["qualified_answer", "abstention"]


class CriticOutput(Strict):
    accepted: bool
    issues: tuple[str, ...] = ()


Claim.model_rebuild(_types_namespace={"Literal": Literal})
Ledger.model_rebuild(_types_namespace={"Claim": Claim})
Synthesis.model_rebuild(_types_namespace={"Literal": Literal})

SCHEMAS = {
    QueryPlan.schema_id: QueryPlan,
    Ledger.schema_id: Ledger,
    Synthesis.schema_id: Synthesis,
    "research.critic.v1": CriticOutput,
}


def fresh_model():
    model_names = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}
    model_path = ROOT / LOCAL_MODEL_PATH if MODEL_PROVIDER == "local" else None
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


def run_coroutine_sync(coroutine):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coroutine)

    result = {}
    error = {}

    def worker():
        try:
            result["value"] = asyncio.run(coroutine)
        except BaseException as exc:  # propagate errors from the adapter thread
            error["value"] = exc

    thread = threading.Thread(target=worker)
    thread.start()
    thread.join()
    if error:
        raise error["value"]
    return result["value"]


class TutorialCrewLLM(BaseLLM):
    """Minimal CrewAI adapter around the repository's provider-neutral model client."""

    def __init__(self):
        model_name = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}[
            MODEL_PROVIDER
        ]
        super().__init__(model=model_name, temperature=0.0)
        self.core = fresh_model()

    def call(
        self,
        messages,
        tools=None,
        callbacks=None,
        available_functions=None,
        from_task=None,
        from_agent=None,
        response_model=None,
        **kwargs,
    ):
        if isinstance(messages, str):
            prompt = messages
        else:
            prompt = "\n".join(
                str(message.get("content", message)) if isinstance(message, dict) else str(message)
                for message in messages
            )
        if response_model is not None:
            schema = response_model
        else:
            schema_id = next((schema_id for schema_id in SCHEMAS if schema_id in prompt), None)
            if schema_id is None:
                raise ValueError(
                    "CrewAI did not provide response_model and the task prompt "
                    "does not include a known SCHEMA identifier."
                )
            schema = SCHEMAS[schema_id]

        response = run_coroutine_sync(self.core.generate([user(prompt)], response_schema=schema))
        return json.dumps(response.structured_output, sort_keys=True)

    def supports_function_calling(self):
        return False

    def supports_stop_words(self):
        return False

    def get_context_window_size(self):
        return 4096


def search(query):
    terms = set(query.casefold().split())
    return [
        record
        for record in catalogue["records"]
        if terms & set((record["title"] + " " + record["passage"]).casefold().split())
    ]


def normalise_tokens(text):
    stopwords = {
        "a",
        "an",
        "and",
        "are",
        "as",
        "at",
        "be",
        "by",
        "for",
        "from",
        "in",
        "is",
        "it",
        "of",
        "on",
        "or",
        "that",
        "the",
        "to",
        "was",
        "were",
        "with",
    }

    def canonicalise(token):
        # Lightweight singularisation is sufficient for the controlled fixture.
        if len(token) > 4 and token.endswith("s"):
            return token[:-1]
        return token

    return {
        canonicalise(token)
        for token in re.findall(r"[a-z0-9]+", text.casefold())
        if token not in stopwords
    }


def claim_grounding_score(claim, record):
    claim_tokens = normalise_tokens(claim.claim)
    passage_tokens = normalise_tokens(record["passage"])
    if len(claim_tokens) < 3:
        return 0.0
    return len(claim_tokens & passage_tokens) / len(claim_tokens)


def claim_is_grounded(claim, record, minimum_overlap=0.5):
    return claim_grounding_score(claim, record) >= minimum_overlap


def build_agents(llm):
    return {
        "planner": Agent(
            role="Research planner",
            goal="Create a small, sufficient set of bounded catalogue queries.",
            backstory="You decompose a research question without assuming the answer.",
            llm=llm,
            allow_delegation=False,
            verbose=False,
        ),
        "extractor": Agent(
            role="Evidence extractor",
            goal="Extract only verbatim claims grounded in supplied records.",
            backstory=(
                "You preserve source identifiers and distinguish supporting from "
                "conflicting evidence."
            ),
            llm=llm,
            allow_delegation=False,
            verbose=False,
        ),
        "writer": Agent(
            role="Evidence synthesiser",
            goal="Answer only from validated claims while preserving conditions and conflicts.",
            backstory="You produce concise, provenance-aware research summaries.",
            llm=llm,
            allow_delegation=False,
            verbose=False,
        ),
        "critic": Agent(
            role="Evidence critic",
            goal="Reject unsupported, unqualified or uncited synthesis.",
            backstory="You apply a strict final quality gate.",
            llm=llm,
            allow_delegation=False,
            verbose=False,
        ),
    }


async def run_typed_task(agent, description, expected_output, output_type):
    task = Task(
        description=description,
        expected_output=expected_output,
        agent=agent,
        output_pydantic=output_type,
    )
    crew = Crew(
        agents=[agent],
        tasks=[task],
        process=Process.sequential,
        verbose=False,
        memory=False,
    )
    with contextlib.redirect_stdout(io.StringIO()):
        result = await crew.kickoff_async()
    if result.pydantic is None:
        raise RuntimeError(f"CrewAI did not return {output_type.__name__}.")
    return output_type.model_validate(result.pydantic)


async def plan(agent, question):
    return await run_typed_task(
        agent,
        (
            f"SCHEMA: {QueryPlan.schema_id}. Create at most four focused catalogue-search "
            f"queries sufficient to answer {question!r}. Do not assume particular interventions."
        ),
        "A QueryPlan JSON object containing one to four focused queries.",
        QueryPlan,
    )


def retrieve(query_plan):
    return {record["source_id"]: record for query in query_plan.queries for record in search(query)}


def screen_evidence(records, safety):
    assessments = {
        source_id: safety.inspect_retrieved_content(
            UntrustedContent(source_id=source_id, text=record["passage"])
        )
        for source_id, record in records.items()
    }
    safe_records = {
        source_id: record
        for source_id, record in records.items()
        if assessments[source_id].decision.outcome in {PolicyOutcome.ALLOW, PolicyOutcome.TRANSFORM}
        and not assessments[source_id].indicators
    }
    isolated = [source_id for source_id, assessment in assessments.items() if assessment.indicators]
    return safe_records, isolated


async def extract_claims(agent, safe_records):
    return await run_typed_task(
        agent,
        (
            f"SCHEMA: {Ledger.schema_id}. Extract one verbatim, source-grounded claim per "
            f"record from {safe_records}. Preserve each source_id. Label reported reductions "
            "as supporting and inconsistent effects as conflicting."
        ),
        "A Ledger JSON object containing grounded claims and source identifiers.",
        Ledger,
    )


def validate_claims(ledger, safe_records):
    return tuple(
        claim
        for claim in ledger.claims
        if claim.source_id in safe_records
        and claim_is_grounded(claim, safe_records[claim.source_id])
    )


async def synthesise(agent, question, claims):
    return await run_typed_task(
        agent,
        (
            f"SCHEMA: {Synthesis.schema_id}. Answer {question!r} using only these validated "
            f"claims: {claims}. State conditions and conflicts, cite source_ids, and use "
            "outcome qualified_answer."
        ),
        "A Synthesis JSON object with an answer, source_ids and outcome.",
        Synthesis,
    )


async def critique(agent, answer, safe_records):
    critic_output = await run_typed_task(
        agent,
        (
            "SCHEMA: research.critic.v1. Accept only if this answer is supported, "
            f"appropriately qualified and cited: {answer.model_dump()}"
        ),
        "A CriticOutput JSON object containing accepted and issues.",
        CriticOutput,
    )
    critic = CriticDecision(
        accepted=critic_output.accepted,
        issues=critic_output.issues,
    )
    citations_valid = set(answer.source_ids).issubset(safe_records)
    return critic, citations_valid


def report(
    *,
    question,
    query_plan,
    retrieved,
    safe_records,
    isolated,
    claims,
    answer=None,
    critic=None,
    citations_valid=False,
    model_calls,
):
    qualified = bool(
        answer is not None
        and critic is not None
        and critic.accepted
        and citations_valid
        and answer.outcome == "qualified_answer"
    )
    outcome = "qualified_answer" if qualified else "abstention"
    if not claims:
        termination = "no_validated_claims"
    elif not citations_valid:
        termination = "invalid_citations"
    elif critic is not None and not critic.accepted:
        termination = "critic_rejected"
    else:
        termination = "criteria_met"

    return {
        "question": question,
        "query_plan": query_plan.model_dump(),
        "claims": claims,
        "answer": answer,
        "outcome": outcome,
        "termination": termination,
        "source_provenance": [] if answer is None else sorted(answer.source_ids),
        "trace": [
            {"event": "plan", "queries": query_plan.queries},
            {"event": "retrieve", "source_ids": sorted(retrieved)},
            {"event": "screen_evidence", "isolated": isolated},
            {"event": "extract_claims", "reported": len(claims)},
            {"event": "validate_claims", "count": len(claims)},
            {"event": "synthesise", "completed": answer is not None},
            {
                "event": "critique",
                "accepted": None if critic is None else critic.accepted,
                "citations_valid": citations_valid,
            },
            {"event": "report", "outcome": outcome},
        ],
        "model_calls": model_calls,
        "safe_source_ids": sorted(safe_records),
    }


async def run_research(question=RESEARCH_QUESTION):
    agents = build_agents(TutorialCrewLLM())
    safety = SafetyEngine(SafetyPolicy(allowed_tools=("catalogue_search",)))

    query_plan = await plan(agents["planner"], question)
    retrieved = retrieve(query_plan)
    safe_records, isolated = screen_evidence(retrieved, safety)
    ledger = await extract_claims(agents["extractor"], safe_records)
    claims = validate_claims(ledger, safe_records)

    if not claims:
        return report(
            question=question,
            query_plan=query_plan,
            retrieved=retrieved,
            safe_records=safe_records,
            isolated=isolated,
            claims=claims,
            model_calls=2,
        )

    answer = await synthesise(agents["writer"], question, claims)
    critic, citations_valid = await critique(agents["critic"], answer, safe_records)
    return report(
        question=question,
        query_plan=query_plan,
        retrieved=retrieved,
        safe_records=safe_records,
        isolated=isolated,
        claims=claims,
        answer=answer,
        critic=critic,
        citations_valid=citations_valid,
        model_calls=4,
    )


# --- Execution: run the workflow and evaluate its observable result ---
first = await run_research(RESEARCH_QUESTION)
second = await run_research(RESEARCH_QUESTION) if MODEL_PROVIDER == "mock" else first
evaluation = {
    "component": len(first["claims"]) == 3,
    "trajectory": len(first["trace"]) == 8 and first["model_calls"] <= 4,
    "task": first["outcome"] == "qualified_answer",
    "safety": "fw-004" not in first["source_provenance"],
    "repeated_run": first == second,
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), {"evaluation": evaluation, "result": first}
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "resource_report": {"model_calls": first["model_calls"], "agents": 4},
    "fallback": "no validated claims or failed critique produces abstention",
}

## Limitation

The custom `BaseLLM` adapter preserves the repository's mock/local/API provider abstraction, while CrewAI supplies the `Agent`, `Task` and `Crew` execution model. The bounded catalogue and deterministic fixture still evaluate orchestration rather than open-domain research quality.
